In [7]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [20]:
import pandas as pd
import numpy as np

df = pd.read_csv("trips.csv")

In [21]:
df.columns

Index(['trip_id', 'date', 'hour', 'zone', 'distance_km', 'duration_min',
       'fare', 'waiting_time_min', 'passengers', 'payment_type', 'weather',
       'driver_supply', 'demand'],
      dtype='object')

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   trip_id            303 non-null    object 
 1   date               303 non-null    object 
 2   hour               303 non-null    int64  
 3   zone               303 non-null    object 
 4   distance_km        303 non-null    float64
 5   duration_min       303 non-null    float64
 6   fare               301 non-null    float64
 7   waiting_time_min   301 non-null    float64
 8   passengers         303 non-null    int64  
 9   payment_type       303 non-null    object 
 10  weather            300 non-null    object 
 11  driver_supply      303 non-null    int64  
 12  demand             303 non-null    int64  
 13  period             303 non-null    object 
 14  supply_demand_gap  303 non-null    int64  
dtypes: float64(4), int64(5), object(6)
memory usage: 35.6+ KB


In [40]:
# Lab 1 — Basic Aggregation on One Column

mean_fare = df["fare"].mean()
median_fare = df["fare"].median()
min_fare = df["fare"].min()
max_fare = df["fare"].max()
std_fare = df["fare"].std()
count_fare = df["fare"].count()

print("Mean:", mean_fare)
print("Median:", median_fare)
print("Minimum:", min_fare)
print("Maximum:", max_fare)
print("Standard deviation:", std_fare)
print("Count:", count_fare)

Mean: 20.304784053156144
Median: 17.31
Minimum: 5.0
Maximum: 132.4
Standard deviation: 13.194635287490076
Count: 301


In [41]:
# you can also apply numpy
np.nanmean(df["fare"])
np.nanmedian(df["fare"])
np.nanmin(df["fare"])
np.nanmax(df["fare"])
np.std(df["fare"])
np.count_nonzero(~np.isnan(df["fare"]))

20.304784053156144

17.31

5.0

132.4

13.172699054053027

301

Interpretation:
The average fare is about $20.30, while the median fare is $17.31. The mean is higher than the median, suggesting that some relatively expensive trips may pull the mean upward.

In [44]:
# Lab 2 — Multiple Aggregations with agg()

wait_summary = df["waiting_time_min"].agg([
    "count",
    "mean",
    "median",
    "std",
    "min",
    "max"
])

print(wait_summary)

count     301.000000
mean        5.179734
median      4.800000
std         3.019518
min         0.800000
max        39.500000
Name: waiting_time_min, dtype: float64


Interpretation:
The mean waiting time is 5.18 minutes, while the median is 4.80 minutes. Since the mean is higher than the median and the maximum is 39.5 minutes, the distribution appears to have some unusually long waiting times and is likely right-skewed.

In [45]:
# Lab 3 — Average Waiting Time by Zone
# single group by, single variable, single function 

zone_wait = (
    df.groupby("zone")["waiting_time_min"]
      .mean()
      .sort_values(ascending=False)
)

print(zone_wait)

zone
Airport        6.103571
University     5.549057
Residential    5.029310
Industrial     4.979310
Downtown       4.639048
Name: waiting_time_min, dtype: float64


In [46]:
zone_wait = (
    df.groupby("zone")["waiting_time_min"]
    .agg(['mean'])
    .sort_values(by='mean',ascending=False)
)

print(zone_wait)

                 mean
zone                 
Airport      6.103571
University   5.549057
Residential  5.029310
Industrial   4.979310
Downtown     4.639048


Airport has the highest average waiting time, approximately 6.10 minutes.

Downtown has the lowest average waiting time, approximately 4.64 minutes.

In [47]:
# Lab 4 — Multiple Statistics by Group
# single group by, single variable, multiple functions

zone_summary = df.groupby("zone")["waiting_time_min"].agg(
    ["count", "mean", "median", "std", "min", "max"]
)

print(zone_summary)

             count      mean  median       std  min   max
zone                                                     
Airport         56  6.103571     5.9  2.052135  3.0  13.5
Downtown       105  4.639048     4.6  1.856511  1.0   9.4
Industrial      29  4.979310     4.9  1.352083  2.8   8.1
Residential     58  5.029310     5.0  1.965760  0.8  11.6
University      53  5.549057     4.2  5.859792  1.1  39.5


Largest number of valid waiting-time observations: Downtown, 105.

Greatest variability: University, because:
        
which is much larger than the other zones.

Highest mean: Airport, 6.10.

Highest median: Airport, 5.90.

University is particularly interesting:
Mean   = 5.55
Median = 4.20
Max    = 39.50

This suggests that one or more unusually large waiting times are pulling the mean upward.

Why report center and variability?
Two groups can have similar average values but very different levels of consistency. The mean or median describes the center, while the standard deviation and range describe how spread out the observations are.

In [48]:
# Lab 5 — Aggregate Multiple Numerical Variables
# single group by, multiple variables, single function

zone_means = df.groupby("zone")[
    ["distance_km", "fare", "waiting_time_min"]
].mean()

print(zone_means)

             distance_km       fare  waiting_time_min
zone                                                 
Airport         7.904821  30.731964          6.103571
Downtown        3.661429  14.962212          4.639048
Industrial      8.439677  27.384516          4.979310
Residential     4.941034  17.441034          5.029310
University      5.130000  18.734231          5.549057


In [49]:
zone_means = df.groupby("zone").agg(
distance_km = ("distance_km", "mean"),
fare = ("fare", "mean"),
waiting_time_min = ("waiting_time_min", "mean")
)

print(zone_means)

             distance_km       fare  waiting_time_min
zone                                                 
Airport         7.904821  30.731964          6.103571
Downtown        3.661429  14.962212          4.639048
Industrial      8.439677  27.384516          4.979310
Residential     4.941034  17.441034          5.029310
University      5.130000  18.734231          5.549057


Longest average distance: Industrial, 8.44 km.

Highest average fare: Airport, $30.73.

Highest average waiting time: Airport, 6.10 minutes.

In [50]:
# Lab 6 — Named Aggregation
# single groupby, multiple variables, multiple functions
zone_summary = df.groupby("zone").agg(
    trip_count=("trip_id", "count"),
    avg_fare=("fare", "mean"),
    median_fare=("fare", "median"),
    avg_wait=("waiting_time_min", "mean"),
    max_wait=("waiting_time_min", "max")
)

zone_summary = zone_summary.sort_values(
    "avg_wait",
    ascending=False
)

print(zone_summary)

             trip_count   avg_fare  median_fare  avg_wait  max_wait
zone                                                               
Airport              56  30.731964       28.830  6.103571      13.5
University           53  18.734231       15.795  5.549057      39.5
Residential          58  17.441034       16.230  5.029310      11.6
Industrial           31  27.384516       22.630  4.979310       8.1
Downtown            105  14.962212       13.710  4.639048       9.4


In [51]:
zone_summary.reset_index()

,zone,trip_count,avg_fare,median_fare,avg_wait,max_wait
0,Airport,56,30.731964,28.830,6.103571,13.5
1,University,53,18.734231,15.795,5.549057,39.5
2,Residential,58,17.441034,16.230,5.029310,11.6
3,Industrial,31,27.384516,22.630,4.979310,8.1
4,Downtown,105,14.962212,13.710,4.639048,9.4


Airport has the highest average waiting time.

In [19]:
# Lab 7 — Group by Two Variables

zone_weather = (
    df.groupby(["zone", "weather"])["waiting_time_min"]
      .agg(["count", "mean", "median"])
)

print(zone_weather)


                     count      mean  median
zone        weather                         
Airport     Clear       31  5.274194    5.20
            Cloudy      15  5.920000    6.00
            Rain         6  8.316667    8.35
            Snow         4  9.900000    9.35
Downtown    Clear       47  3.653191    3.80
            Cloudy      27  4.207407    4.40
            Rain        19  6.436842    6.30
            Snow        11  6.763636    7.10
Industrial  Clear       21  4.847619    4.50
            Cloudy       7  5.071429    5.20
            Rain         1  7.100000    7.10
Residential Clear       28  4.171429    4.10
            Cloudy      20  4.975000    5.00
            Rain         7  6.800000    6.70
            Snow         3  9.266667    8.80
University  Clear       23  4.508696    3.40
            Cloudy      12  3.725000    3.40
            Rain        10  9.380000    6.70
            Snow         6  7.216667    7.60


Highest mean:
Airport + Snow = 9.90 minutes

In [22]:
# Lab 8 — Peak vs Off-Peak
def get_period(hour):
    if hour in [7, 8, 9, 16, 17, 18]:
        return "Peak"
    else:
        return "Off-Peak"

df["period"] = df["hour"].apply(get_period)


period_summary = df.groupby("period").agg(
    trip_count=("trip_id", "count"),
    avg_fare=("fare", "mean"),
    avg_wait=("waiting_time_min", "mean"),
    median_wait=("waiting_time_min", "median"),
    avg_demand=("demand", "mean"),
    avg_supply=("driver_supply", "mean")
)

print(period_summary)

          trip_count   avg_fare  avg_wait  median_wait  avg_demand  avg_supply
period                                                                        
Off-Peak         207  19.489122  5.152913         4.75   83.173913   87.632850
Peak              96  22.046562  5.237895         5.10  103.052083   95.979167


Both demand and driver supply are higher during peak periods. However, demand increases more strongly than supply, and waiting times are slightly higher during peak periods.

In [23]:
# Lab 9 — Custom Aggregation Function

# Function：
def data_range(x):
    return x.max() - x.min()

# Overall fare range：
df["fare"].agg(data_range)

# By zone：
df.groupby("zone")["fare"].agg(data_range)

# Waiting-time range：
df.groupby("zone")["waiting_time_min"].agg(data_range)

127.4

zone
Airport         43.60
Downtown        34.26
Industrial     122.87
Residential     35.41
University     112.24
Name: fare, dtype: float64

zone
Airport        10.5
Downtown        8.4
Industrial      5.3
Residential    10.8
University     38.4
Name: waiting_time_min, dtype: float64

University has the largest waiting-time range: 38.4 minutes.

In [24]:
# Lab 10 — Aggregation After Filtering

# Filter：

longer_waits = df[
    df["waiting_time_min"] > 5
]

# Aggregation：

filtered_summary = longer_waits.groupby("zone")[
    "waiting_time_min"
].agg(["count", "mean"])

print(filtered_summary)



             count      mean
zone                        
Airport         36  7.194444
Downtown        40  6.527500
Industrial      13  6.223077
Residential     29  6.565517
University      20  9.460000


Filtering changes the population being summarized. Therefore, an average calculated after filtering answers a different question from the average calculated using the full dataset.

In [25]:
# Lab 11 — groupby() + sort_values()

zone_rank = df.groupby("zone").agg(
    trip_count=("trip_id", "count"),
    avg_fare=("fare", "mean"),
    avg_wait=("waiting_time_min", "mean")
)

zone_rank = zone_rank.sort_values(
    "avg_wait",
    ascending=False
)

print(zone_rank)

# Top 3:
zone_rank.head(3)

# Sort by trip count：
zone_rank.sort_values(
    "trip_count",
    ascending=False
)


             trip_count   avg_fare  avg_wait
zone                                        
Airport              56  30.731964  6.103571
University           53  18.734231  5.549057
Residential          58  17.441034  5.029310
Industrial           31  27.384516  4.979310
Downtown            105  14.962212  4.639048


,trip_count,avg_fare,avg_wait
zone,,,
Airport,56,30.731964,6.103571
University,53,18.734231,5.549057
Residential,58,17.441034,5.029310


,trip_count,avg_fare,avg_wait
zone,,,
Downtown,105,14.962212,4.639048
Residential,58,17.441034,5.029310
Airport,56,30.731964,6.103571
University,53,18.734231,5.549057
Industrial,31,27.384516,4.979310


Rankings depend on the metric being used. “Busiest” and “slowest service” are different concepts.

In [26]:
# Lab 12 — Pivot Table
# Mean:
    
pivot_mean = pd.pivot_table(
    df,
    values="waiting_time_min",
    index="zone",
    columns="weather",
    aggfunc="mean"
)

print(pivot_mean)

# Overall weather means：
df.groupby("weather")["waiting_time_min"].mean()


# Median pivot：
pivot_median = pd.pivot_table(
    df,
    values="waiting_time_min",
    index="zone",
    columns="weather",
    aggfunc="median"
)


weather         Clear    Cloudy      Rain      Snow
zone                                               
Airport      5.274194  5.920000  8.316667  9.900000
Downtown     3.653191  4.207407  6.436842  6.763636
Industrial   4.847619  5.071429  7.100000       NaN
Residential  4.171429  4.975000  6.800000  9.266667
University   4.508696  3.725000  9.380000  7.216667


weather
Clear     4.383333
Cloudy    4.717284
Rain      7.458140
Snow      7.712500
Name: waiting_time_min, dtype: float64

In [27]:
# Lab 13 — Pivot Table with Mean and Median

pivot_both = pd.pivot_table(
    df,
    values="waiting_time_min",
    index="zone",
    columns="weather",
    aggfunc=["mean", "median"]
)

print(pivot_both)

                 mean                               median                   
weather         Clear    Cloudy      Rain      Snow  Clear Cloudy  Rain  Snow
zone                                                                         
Airport      5.274194  5.920000  8.316667  9.900000    5.2    6.0  8.35  9.35
Downtown     3.653191  4.207407  6.436842  6.763636    3.8    4.4  6.30  7.10
Industrial   4.847619  5.071429  7.100000       NaN    4.5    5.2  7.10   NaN
Residential  4.171429  4.975000  6.800000  9.266667    4.1    5.0  6.70  8.80
University   4.508696  3.725000  9.380000  7.216667    3.4    3.4  6.70  7.60


In [28]:
# Lab 14 — Supply and Demand

df["supply_demand_gap"] = (
    df["driver_supply"] - df["demand"]
)


supply_summary = df.groupby("zone").agg(
    avg_demand=("demand", "mean"),
    avg_supply=("driver_supply", "mean"),
    avg_gap=("supply_demand_gap", "mean"),
    avg_wait=("waiting_time_min", "mean")
)

print(supply_summary)



             avg_demand  avg_supply   avg_gap  avg_wait
zone                                                   
Airport       92.535714   89.446429 -3.089286  6.103571
Downtown     100.666667  103.466667  2.800000  4.639048
Industrial    71.451613   74.354839  2.903226  4.979310
Residential   79.224138   82.172414  2.948276  5.029310
University    85.811321   83.207547 -2.603774  5.549057


In [29]:
# Final Aggregation Challenge

final_summary = df.groupby("zone").agg(
    trip_count=("trip_id", "count"),
    avg_distance=("distance_km", "mean"),
    avg_fare=("fare", "mean"),
    median_fare=("fare", "median"),
    avg_wait=("waiting_time_min", "mean"),
    median_wait=("waiting_time_min", "median"),
    wait_std=("waiting_time_min", "std"),
    wait_iqr=(
        "waiting_time_min",
        lambda x: x.quantile(0.75) - x.quantile(0.25)
    ),
    avg_demand=("demand", "mean"),
    avg_supply=("driver_supply", "mean")
)

final_summary = (
    final_summary
    .sort_values("avg_wait", ascending=False)
    .round(2)
)

print(final_summary)


             trip_count  avg_distance  avg_fare  median_fare  avg_wait  \
zone                                                                     
Airport              56          7.90     30.73        28.83      6.10   
University           53          5.13     18.73        15.80      5.55   
Residential          58          4.94     17.44        16.23      5.03   
Industrial           31          8.44     27.38        22.63      4.98   
Downtown            105          3.66     14.96        13.71      4.64   

             median_wait  wait_std  wait_iqr  avg_demand  avg_supply  
zone                                                                  
Airport              5.9      2.05      2.73       92.54       89.45  
University           4.2      5.86      3.70       85.81       83.21  
Residential          5.0      1.97      2.27       79.22       82.17  
Industrial           4.9      1.35      2.20       71.45       74.35  
Downtown             4.6      1.86      2.60      100.6

Downtown has the largest number of trips, but it has the lowest average waiting time among the five zones. Airport has the highest average fare and the highest average waiting time. University shows the greatest variability in waiting time, with both the largest standard deviation and IQR, partly because of several unusually long waits. Overall, the results show that trip volume, fare, waiting time, and variability differ substantially across zones.